# 02 · Instructions are the only orchestration lever, and the eval harness

## Goal

Create the real spine agent — Contract Renewal Desk — with a first cut of
`instructions.md`, and introduce `run_suite()`: the same call every notebook
from here to `25` ends with. Leave this notebook with a green 12-case
golden set and a habit: change something, run the suite, read the diff.


## Prereqs

Asserted below, not just stated — this cell fails loudly if a prior notebook's step wasn't actually completed.


In [ ]:
from csx.config import load_settings
from csx.pac import copilot_init
from pathlib import Path

settings = load_settings()
settings.require("DATAVERSE_ENV_ID", "APP_CLIENT_ID", "DELEGATED_CLIENT_ID")

workspace = Path("../agents/contract-renewal-desk")
assert workspace.exists(), "Run from notebooks/ with the repo layout intact"
assert (workspace / "instructions.md").exists(), "agents/contract-renewal-desk/instructions.md missing"
print("spine workspace found:", workspace)


## Concept

**Finding #3 reshapes this notebook entirely.** On the GHCP harness,
orchestration is not configurable — one enhanced orchestration model
handles every agent, full stop. There is no orchestrator to tune. The
levers you actually have are: instructions, skill descriptions, tool
descriptions, and knowledge-source descriptions. This notebook is titled
around the one of those four available from day one — instructions — and
every later notebook that touches selection behaviour (`06`, `08`, `11`)
is really still working this same lever, just for a different surface.

**Why the eval harness lands here, not in notebook 23:** a notebook that
says "the instructions feel better now" has taught nothing measurable.
`run_suite()` sends the 12-case golden set in `evals/golden_cases.json` to
the published agent and grades each response. Every notebook from here on
ends with the same call. When a later notebook's new knowledge source or
workflow branch regresses something instructions used to get right, you'll
see it here — not in a support ticket three months into production.


## Build


### First real instructions

`agents/contract-renewal-desk/instructions.md` already has a full draft (see the file — written for this notebook). Read it, then push it.


In [ ]:
print((workspace / "instructions.md").read_text())


In [ ]:
from csx.pac import copilot_push
copilot_push(workspace)
# publish explicitly — push alone doesn't publish
import subprocess
subprocess.run(["pac", "copilot", "publish", "--name", "crd_contract-renewal-desk"], check=True)


## Verify

Same harness, same golden set, every notebook.


In [ ]:
from csx.clients import get_copilot_client
from csx.verify import run_suite, load_golden
from csx.cost import CreditMeter

client = get_copilot_client(settings, delegated=True)
meter = CreditMeter(environment_id=settings.get("DATAVERSE_ENV_ID"))

core_cases = load_golden(tags=["core"])
assert len(core_cases) == 12, f"expected the 12-case core set, found {len(core_cases)}"

suite = run_suite(client, cases=core_cases, credit_meter=meter, min_pass_rate=0.8)


If this failed: read the `reason` column, not just the pass rate. Most first-pass failures here are the agent answering out-of-scope questions instead of declining (`core-02`, `core-05`) — tighten the scope section in `instructions.md`, push, re-run this cell. Don't move on until it's green; every later notebook assumes this floor holds.


## Cost


In [ ]:
meter.report_cost("02", budget=settings.get("COPILOT_CREDIT_BUDGET"), delta_credits=suite.total_credits, note="instructions iteration + first golden run")


## Teardown


In [ ]:
print("No teardown — this is the spine agent. It persists through notebook 25.")
